# DC Motor Energy Function

This notebook is the DC motor equivalent of `TestEnergyFunction.ipynb`.

For the no-input DC motor, a natural physical energy proxy is:

```text
E(omega, current) = 0.5 * J * omega^2 + 0.5 * L * current^2
```

For the default parameters with `Kt = Ke`, `Va = 0`, and `Tl = 0`, its derivative along the true dynamics is non-positive:

```text
dE/dt = -b * omega^2 - R * current^2
```

That makes it a useful reference Lyapunov-like function for checking whether the DC motor benchmark is dissipative.

In [ ]:
%matplotlib inline
import glob
import matplotlib.pyplot as plt
import numpy as np
import torch

from datasets.dc_motor import DEFAULTS, dc_motor_gradient
from util import DynamicLoad, latest_file

## Parameters

The default DC motor benchmark is autonomous because `Va=0` and `Tl=0`.
The origin `[omega, current] = [0, 0]` is the equilibrium.

In [ ]:
params = DEFAULTS.copy()
params.update({
    "Va": 0.0,
    "Tl": 0.0,
})

rhs = dc_motor_gradient(params)

for k, v in params.items():
    print(f"{k:>12s}: {v}")

## Energy and energy derivative

We compare two equivalent ways of computing `dE/dt`:

1. Analytic expression: `dE/dt = J * omega * omega_dot + L * current * current_dot`
2. Simplified dissipative expression for `Va=0`, `Tl=0`, and `Kt=Ke`: `dE/dt = -b * omega^2 - R * current^2`

In [ ]:
def motor_energy(x, params):
    x = np.asarray(x, dtype=np.float32)
    omega = x[..., 0]
    current = x[..., 1]
    return 0.5 * params["J"] * omega**2 + 0.5 * params["L"] * current**2

def motor_energy_derivative(x, params):
    x = np.asarray(x, dtype=np.float32)
    dx = rhs(x)
    omega = x[..., 0]
    current = x[..., 1]
    omega_dot = dx[..., 0]
    current_dot = dx[..., 1]
    return params["J"] * omega * omega_dot + params["L"] * current * current_dot

def motor_energy_derivative_simplified(x, params):
    x = np.asarray(x, dtype=np.float32)
    omega = x[..., 0]
    current = x[..., 1]
    return -params["b"] * omega**2 - params["R"] * current**2

test_points = np.array([
    [0.0, 0.0],
    [1.0, 0.0],
    [0.0, 1.0],
    [2.0, -3.0],
    [-4.0, 2.0],
], dtype=np.float32)

print("x, E(x), dE/dt, simplified dE/dt")
for x in test_points:
    print(x, motor_energy(x, params), motor_energy_derivative(x, params), motor_energy_derivative_simplified(x, params))

diff = motor_energy_derivative(test_points, params) - motor_energy_derivative_simplified(test_points, params)
print("\nmax absolute difference:", np.max(np.abs(diff)))

## Energy surface

The energy is a positive bowl centered at the origin. It is zero only at the equilibrium.

In [ ]:
omega_grid = np.linspace(-5.0, 5.0, 101)
current_grid = np.linspace(-5.0, 5.0, 101)
Omega, Current = np.meshgrid(omega_grid, current_grid)
states = np.stack([Omega.ravel(), Current.ravel()], axis=1).astype(np.float32)

Energy = motor_energy(states, params).reshape(Omega.shape)

plt.figure(figsize=(7, 6))
contour = plt.contourf(Omega, Current, Energy, levels=40)
plt.scatter([0], [0], marker="x", s=100)
plt.xlabel("omega [rad/s]")
plt.ylabel("current [A]")
plt.title("DC motor physical energy proxy")
plt.colorbar(contour, label="E(omega, current)")
plt.tight_layout()
plt.show()

## Energy derivative surface

For the no-input default system, this should be non-positive over the whole grid. That is the basic stability check performed in this notebook.

In [ ]:
dEnergy = motor_energy_derivative(states, params).reshape(Omega.shape)

print("max dE/dt over grid:", np.max(dEnergy))
print("min dE/dt over grid:", np.min(dEnergy))
print("all non-positive:", np.all(dEnergy <= 1e-7))

plt.figure(figsize=(7, 6))
contour = plt.contourf(Omega, Current, dEnergy, levels=40)
plt.scatter([0], [0], marker="x", s=100)
plt.xlabel("omega [rad/s]")
plt.ylabel("current [A]")
plt.title("Energy derivative along DC motor dynamics")
plt.colorbar(contour, label="dE/dt")
plt.tight_layout()
plt.show()

## Vector field over energy contours

The arrows show the physical dynamics, while the contours show energy. For a dissipative autonomous motor, trajectories should move toward lower energy.

In [ ]:
coarse = np.linspace(-5.0, 5.0, 21)
Om_c, Cur_c = np.meshgrid(coarse, coarse)
states_c = np.stack([Om_c.ravel(), Cur_c.ravel()], axis=1).astype(np.float32)
derivatives_c = rhs(states_c)
dOm = derivatives_c[:, 0].reshape(Om_c.shape)
dCur = derivatives_c[:, 1].reshape(Cur_c.shape)
speed = np.sqrt(dOm**2 + dCur**2)

dOm_n = dOm / np.maximum(speed, 1e-8)
dCur_n = dCur / np.maximum(speed, 1e-8)

plt.figure(figsize=(7, 6))
plt.contour(Omega, Current, Energy, levels=20)
plt.quiver(Om_c, Cur_c, dOm_n, dCur_n, speed)
plt.scatter([0], [0], marker="x", s=100)
plt.xlabel("omega [rad/s]")
plt.ylabel("current [A]")
plt.title("Vector field over energy contours")
plt.colorbar(label="||f(x)||")
plt.tight_layout()
plt.show()

## Energy decay during solver simulation

We now simulate a few trajectories and verify that the solver energy decreases over time.

In [ ]:
def rk4_step(rhs, x, dt):
    k1 = rhs(x)
    k2 = rhs(x + 0.5 * dt * k1)
    k3 = rhs(x + 0.5 * dt * k2)
    k4 = rhs(x + dt * k3)
    return x + (dt / 6.0) * (k1 + 2.0 * k2 + 2.0 * k3 + k4)

def simulate(rhs, x0, steps=400, dt=0.01):
    x = np.asarray(x0, dtype=np.float32)
    trajectory = np.zeros((steps, *x.shape), dtype=np.float32)
    trajectory[0] = x
    for step in range(1, steps):
        x = rk4_step(rhs, x, dt).astype(np.float32)
        trajectory[step] = x
    return trajectory

initial_states = np.array([
    [ 4.0,  4.0],
    [ 4.0, -4.0],
    [-4.0,  4.0],
    [-4.0, -4.0],
    [ 2.0,  0.0],
    [ 0.0,  2.0],
], dtype=np.float32)

dt = 0.01
steps = 400
time = np.arange(steps) * dt
trajectories = simulate(rhs, initial_states, steps=steps, dt=dt)

energies = motor_energy(trajectories, params)

print("initial energies:", energies[0])
print("final energies:", energies[-1])
print("energy decreased for every path:", np.all(energies[-1] <= energies[0]))

In [ ]:
plt.figure(figsize=(8, 4))
for idx in range(energies.shape[1]):
    plt.plot(time, energies[:, idx], label=f"path {idx}")
plt.xlabel("time [s]")
plt.ylabel("E(omega, current)")
plt.yscale("log")
plt.title("Energy decay in no-input DC motor")
plt.tight_layout()
plt.show()

plt.figure(figsize=(6, 6))
for idx in range(trajectories.shape[1]):
    plt.plot(trajectories[:, idx, 0], trajectories[:, idx, 1], label=f"path {idx}")
    plt.scatter(trajectories[0, idx, 0], trajectories[0, idx, 1], marker="o")
plt.scatter([0], [0], marker="x", s=100)
plt.xlabel("omega [rad/s]")
plt.ylabel("current [A]")
plt.title("Phase portrait with decreasing energy")
plt.legend()
plt.tight_layout()
plt.show()

# Long-horizon energy: solver vs trained model

This section loads a trained DC motor dynamics model and compares its long-horizon energy behavior against the physical solver.

Run `bash train_dc_motor_stable` first, or change `MODEL_SPEC` and `WEIGHT_GLOB` below.

In [ ]:
MODEL_SPEC = "stabledynamics[latent_space_dim=2,a=0.001,projfn=NN-REHU,projfn_eps=0.01,smooth_v=0,hp=60,h=100,rehu=0.001]"
WEIGHT_GLOB = "experiments/dc-motor-stable/checkpoint_*.pth"
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

try:
    weight_path = latest_file(WEIGHT_GLOB)
    model_module = DynamicLoad("models")(MODEL_SPEC)
    model = model_module.model
    model.load_state_dict(torch.load(weight_path, map_location=DEVICE))
    model.to(DEVICE)
    model.eval()
    MODEL_AVAILABLE = True
    print(f"Loaded model from {weight_path} on {DEVICE}")
except Exception as exc:
    MODEL_AVAILABLE = False
    model = None
    weight_path = None
    print("No trained checkpoint available yet.")
    print("Train one with: bash train_dc_motor_stable")
    print("Original error:", repr(exc))

In [ ]:
def rk4_model_step(model, x, dt):
    # stabledynamics needs gradients wrt x, so do not wrap this in torch.no_grad().
    x = x.detach().requires_grad_(True)
    k1 = model(x)
    k2 = model((x + 0.5 * dt * k1).detach().requires_grad_(True))
    k3 = model((x + 0.5 * dt * k2).detach().requires_grad_(True))
    k4 = model((x + dt * k3).detach().requires_grad_(True))
    return (x + (dt / 6.0) * (k1 + 2.0 * k2 + 2.0 * k3 + k4)).detach()

def simulate_model(model, x0, steps=2000, dt=0.01):
    x = torch.tensor(x0, dtype=torch.float32, device=DEVICE)
    trajectory = np.zeros((steps, *x0.shape), dtype=np.float32)
    trajectory[0] = x.detach().cpu().numpy()
    for step in range(1, steps):
        x = rk4_model_step(model, x, dt)
        trajectory[step] = x.detach().cpu().numpy().astype(np.float32)
    return trajectory

In [ ]:
if MODEL_AVAILABLE:
    long_x0 = np.array([
        [ 5.0,  5.0],
        [ 5.0, -5.0],
        [-5.0,  5.0],
        [-5.0, -5.0],
        [ 3.0,  0.0],
        [ 0.0,  3.0],
        [ 1.0, -4.0],
        [-2.0,  4.0],
    ], dtype=np.float32)

    long_steps = 2000
    long_dt = 0.01
    long_time = np.arange(long_steps) * long_dt

    solver_long = simulate(rhs, long_x0, steps=long_steps, dt=long_dt)
    model_long = simulate_model(model, long_x0, steps=long_steps, dt=long_dt)

    solver_energy = motor_energy(solver_long, params)
    model_energy = motor_energy(model_long, params)

    mean_solver_energy = np.mean(solver_energy, axis=1)
    mean_model_energy = np.mean(model_energy, axis=1)
    energy_error = mean_model_energy - mean_solver_energy

    model_energy_increases = np.diff(model_energy, axis=0) > 1e-7

    print("Long-horizon energy comparison complete")
    print("horizon seconds:", long_steps * long_dt)
    print("solver final mean energy:", mean_solver_energy[-1])
    print("model final mean energy:", mean_model_energy[-1])
    print("mean absolute energy error:", np.mean(np.abs(energy_error)))
    print("number of model per-path energy increases:", int(np.sum(model_energy_increases)))
else:
    print("Skipping long-horizon energy comparison until a checkpoint exists.")

## Mean energy over long horizon

The solver should monotonically decay to zero. A good stable model should track that decay and avoid artificial energy growth.

In [ ]:
if MODEL_AVAILABLE:
    plt.figure(figsize=(9, 4))
    plt.semilogy(long_time, mean_solver_energy, label="solver mean energy")
    plt.semilogy(long_time, mean_model_energy, linestyle="--", label="model mean energy")
    plt.xlabel("time [s]")
    plt.ylabel("mean E(omega, current)")
    plt.title("Long-horizon mean energy: solver vs model")
    plt.legend()
    plt.tight_layout()
    plt.show()

## Per-trajectory energy comparison

Solid lines are solver energy. Dashed lines are model energy for the same initial states.

In [ ]:
if MODEL_AVAILABLE:
    plt.figure(figsize=(10, 5))
    for idx in range(long_x0.shape[0]):
        plt.semilogy(long_time, solver_energy[:, idx], linestyle="-", linewidth=1.0)
        plt.semilogy(long_time, model_energy[:, idx], linestyle="--", linewidth=1.0)
    plt.xlabel("time [s]")
    plt.ylabel("E(omega, current)")
    plt.title("Per-trajectory energy: solver solid vs model dashed")
    plt.tight_layout()
    plt.show()

## Energy error

This shows how much physical energy the learned model has relative to the solver over time.

In [ ]:
if MODEL_AVAILABLE:
    plt.figure(figsize=(9, 4))
    plt.plot(long_time, energy_error, label="model mean energy - solver mean energy")
    plt.axhline(0.0, linewidth=1.0)
    plt.xlabel("time [s]")
    plt.ylabel("energy difference")
    plt.title("Long-horizon energy error")
    plt.legend()
    plt.tight_layout()
    plt.show()

## Model energy monotonicity diagnostic

This plot counts how many trajectories experience an energy increase at each step. For a perfectly dissipative model, this would remain zero. Small nonzero values indicate local violations of the physical energy decrease, even if the learned Lyapunov function may still be decreasing.

In [ ]:
if MODEL_AVAILABLE:
    increases_per_step = np.sum(model_energy_increases, axis=1)
    plt.figure(figsize=(9, 3))
    plt.plot(long_time[1:], increases_per_step)
    plt.xlabel("time [s]")
    plt.ylabel("# paths with E increase")
    plt.title("Physical energy increase diagnostic for learned model")
    plt.tight_layout()
    plt.show()